In [2]:
import pandas as pd, numpy as np

df=pd.read_excel("Raw-datasets/thinning-data-2.xlsx")
d=df.copy()
for col in ['DBH','HT']:
    d[col]=pd.to_numeric(d[col].astype(str).str.replace('\xa0','').str.strip(),errors='coerce')
d['status']=np.where(d['DBH'].notna() & d['HT'].notna(),'Alive','Dead')
d=d.drop(columns=['RowA','Codes'],errors='ignore')
d.to_csv("clean-thinning_data-2.csv",index=False)  # new file


In [14]:
import pandas as pd, numpy as np

def standardize_for_algorithms_keep_dists(in_path, out_path, row_space_ft=10.0, col_space_ft=10.0, treat_zero_as_dead=False):
    d=pd.read_csv(in_path)
    d=d.rename(columns=lambda c:str(c).strip())

    if 'Tree' in d.columns and 'TREE #' in d.columns:
        d=d.rename(columns={'Tree':'tree_index'})

    # Standard renames
    ren={'ROW':'Row','TREE #':'Tree','DBH':'DBH_in','HT':'HT_ft','CuFT':'volume_ft3','CuMe':'volume_m3','HTLC':'HTLC_ft','Codes':'codes'}
    d=d.rename(columns={k:v for k,v in ren.items() if k in d.columns})

    lower={c.lower():c for c in d.columns}
    if 'dist from tree 1 ft' in lower:
        d=d.rename(columns={lower['dist from tree 1 ft']:'dist_from_tree1_ft'})
    if 'dist from tree 1' in lower:
        if lower['dist from tree 1']!=lower.get('dist from tree 1 ft',None):
            d=d.rename(columns={lower['dist from tree 1']:'dist_from_tree1'})

    dup_names=d.columns[d.columns.duplicated(keep=False)].unique()
    for nm in dup_names:
        if nm in ('dist_from_tree1','dist_from_tree1_ft'): continue
        blk=d.loc[:,d.columns==nm]
        if blk.shape[1]>1:
            d[nm]=blk.bfill(axis=1).iloc[:,0]
            d.drop(columns=blk.columns[1:],inplace=True)

    # Types
    if 'Row' in d.columns:  d['Row']=pd.to_numeric(d['Row'],errors='coerce').astype('Int64')
    if 'Tree' in d.columns: d['Tree']=pd.to_numeric(d['Tree'],errors='coerce').astype('Int64')
    for c in ['DBH_in','HT_ft','HTLC_ft','volume_ft3','volume_m3','dist_from_tree1','dist_from_tree1_ft']:
        if c in d.columns: d[c]=pd.to_numeric(d[c],errors='coerce')

    if 'DBH_in' not in d.columns: d['DBH_in']=np.nan
    if 'HT_ft'  not in d.columns: d['HT_ft']=np.nan
    ok=d['DBH_in'].notna() & d['HT_ft'].notna()
    if treat_zero_as_dead: ok=ok & (d['DBH_in']>0) & (d['HT_ft']>0)
    d['status']=np.where(ok,'Alive','Dead')

    d=d.drop(columns=['RowA'],errors='ignore')

    d['x_ft']=(d['Tree'].astype(float)-1)*col_space_ft
    d['y_ft']=(d['Row'].astype(float)-1)*row_space_ft

    base=['Row','Tree','DBH_in','HT_ft','HTLC_ft','dist_from_tree1','dist_from_tree1_ft','tree_index','volume_ft3','volume_m3','status','x_ft','y_ft']
    d=d[[c for c in base if c in d.columns]+[c for c in d.columns if c not in base]]

    d.to_csv(out_path,index=False)
    print(f"Saved {out_path} | rows={len(d)} alive={(d['status']=='Alive').sum()} dead={(d['status']=='Dead').sum()}")
    return out_path

standardize_for_algorithms_keep_dists("clean-thinning_data-2.csv","clean-thinning_data-2_std.csv")


Saved clean-thinning_data-2_std.csv | rows=3000 alive=2595 dead=405


'clean-thinning_data-2_std.csv'

In [16]:
import pandas as pd, numpy as np

def make_legacy_schema(in_path, out_path, row_space_ft=10.0, col_space_ft=10.0):
    d=pd.read_csv(in_path)
    d=d.rename(columns=lambda c:str(c).strip())

    # --- Helpers ---
    def first_existing(cols):
        for c in cols:
            if c in d.columns: return c
        return None

    # Unify Row/Tree names
    if 'Row' not in d.columns and 'ROW' in d.columns: d=d.rename(columns={'ROW':'Row'})
    if 'Tree' not in d.columns and 'TREE #' in d.columns: d=d.rename(columns={'TREE #':'Tree'})
    d['Row']=pd.to_numeric(d.get('Row'),errors='coerce').astype('Int64')
    d['Tree']=pd.to_numeric(d.get('Tree'),errors='coerce').astype('Int64')

    src_DBH=first_existing(['pre_DBH','DBH_in','DBH'])
    src_HT =first_existing(['pre_HT','HT_ft','HT'])
    src_vol=first_existing(['pre_stem_vol','volume_ft3','CuFT'])
    src_volm3=first_existing(['pre_stem_vol_m3','volume_m3','CuMe'])
    src_x  =first_existing(['x','x_ft'])
    src_y  =first_existing(['y','y_ft'])

    if src_DBH: d['pre_DBH']=pd.to_numeric(d[src_DBH],errors='coerce')
    else: d['pre_DBH']=np.nan
    if src_HT: d['pre_HT']=pd.to_numeric(d[src_HT],errors='coerce')
    else: d['pre_HT']=np.nan
    if src_vol: d['pre_stem_vol']=pd.to_numeric(d[src_vol],errors='coerce')
    else: d['pre_stem_vol']=0.0
    if src_volm3: d['pre_stem_vol_m3']=pd.to_numeric(d[src_volm3],errors='coerce')

    # Coordinates
    if src_x: d['x']=pd.to_numeric(d[src_x],errors='coerce')
    else: d['x']=(d['Tree'].astype(float)-1.0)*col_space_ft
    if src_y: d['y']=pd.to_numeric(d[src_y],errors='coerce')
    else: d['y']=(d['Row'].astype(float)-1.0)*row_space_ft

    # Keep BOTH distance columns
    if 'DIST from Tree 1' not in d.columns and 'dist_from_tree1' in d.columns:
        d['DIST from Tree 1']=pd.to_numeric(d['dist_from_tree1'],errors='coerce')
    if 'Dist from Tree 1 Ft' not in d.columns and 'dist_from_tree1_ft' in d.columns:
        d['Dist from Tree 1 Ft']=pd.to_numeric(d['dist_from_tree1_ft'],errors='coerce')

    # Status
    if 'status' not in d.columns:
        ok = d['pre_DBH'].notna() & d['pre_HT'].notna()
        d['status']=np.where(ok,'Alive','Dead')

    d=d.drop(columns=['RowA'],errors='ignore')

    base=['Row','Tree','pre_DBH','pre_HT','pre_stem_vol','pre_stem_vol_m3',
          'status','x','y','DIST from Tree 1','Dist from Tree 1 Ft']
    ordered=[c for c in base if c in d.columns] + [c for c in d.columns if c not in base]
    d=d[ordered]

    d.to_csv(out_path,index=False)
    print(f"Saved {out_path} | rows={len(d)} | Alive={(d['status']=='Alive').sum()} | Dead={(d['status']=='Dead').sum()}")

make_legacy_schema("data/clean-thinning-dataset-2.csv","data/clean-thinning_data-2_std.csv")


Saved data/clean-thinning_data-2_std.csv | rows=3000 | Alive=2595 | Dead=405


### Cleaning dataset-3

In [19]:
import pandas as pd, numpy as np, pathlib

def clean_thinning_v3_fixed(input_path="thinning-data-3.xlsx",
                            output_path="thinning-data-3_std.csv",
                            treat_zero_as_dead=False):
    p=pathlib.Path(input_path); ext=p.suffix.lower()
    if ext==".csv": d=pd.read_csv(p)
    elif ext in (".xlsx",".xls"): d=pd.read_excel(p)
    else: raise ValueError("Use .csv or .xlsx input")

    # --- normalize headers ---
    d=d.rename(columns=lambda c:str(c).strip())

    # --- Row column---
    if 'Row' not in d.columns and 'ROW' in d.columns: d=d.rename(columns={'ROW':'Row'})
    d['Row']=pd.to_numeric(d.get('Row'),errors='coerce').astype('Int64')

    if 'Tree123' in d.columns: d=d.drop(columns=['Tree123'])
    d['Tree']=d.groupby('Row').cumcount()+1
    d['Tree']=d['Tree'].astype('Int64')

    # --- measurements ---
    def _num(series):
        s=pd.Series(series).astype(str).str.replace('\xa0','',regex=False).str.strip()
        s=s.replace({'X':np.nan,'x':np.nan,'':np.nan,'nan':np.nan,'None':np.nan})
        return pd.to_numeric(s,errors='coerce')

    # DBH / HT
    src_dbh = next((c for c in ['pre_DBH','DBH','DBH_in'] if c in d.columns), None)
    src_ht  = next((c for c in ['pre_HT','HT','HT_ft'] if c in d.columns), None)
    d['pre_DBH']=_num(d[src_dbh]) if src_dbh else np.nan
    d['pre_HT'] =_num(d[src_ht ]) if src_ht  else np.nan

    # volumes
    if 'VolCuFT' in d.columns: d=d.rename(columns={'VolCuFT':'pre_stem_vol'})
    if 'VolCuMe' in d.columns: d=d.rename(columns={'VolCuMe':'pre_stem_vol_m3'})
    for c in ['pre_stem_vol','pre_stem_vol_m3']:
        if c in d.columns: d[c]=pd.to_numeric(d[c],errors='coerce')

    # --- keep distance columns if present ---
    lower={c.lower():c for c in d.columns}
    if 'dist from tree 1' in lower and 'DIST from Tree 1' not in d.columns:
        d=d.rename(columns={lower['dist from tree 1']:'DIST from Tree 1'})
    if 'dist from tree 1 ft' in lower and 'Dist from Tree 1 Ft' not in d.columns:
        d=d.rename(columns={lower['dist from tree 1 ft']:'Dist from Tree 1 Ft'})

    # --- status---
    ok = d['pre_DBH'].notna() & d['pre_HT'].notna()
    if treat_zero_as_dead:
        ok = ok & (d['pre_DBH']>0) & (d['pre_HT']>0)
    d['status']=np.where(ok,'Alive','Dead')

    d=d.drop(columns=[c for c in ['x','y','x_ft','y_ft'] if c in d.columns], errors='ignore')

    d=d.drop(columns=['RowA'],errors='ignore')
    base=['Row','Tree','pre_DBH','pre_HT','pre_stem_vol','pre_stem_vol_m3',
          'status','DIST from Tree 1','Dist from Tree 1 Ft']
    d=d[[c for c in base if c in d.columns]+[c for c in d.columns if c not in base]]

    d.to_csv(output_path,index=False)
    alive=(d['status']=='Alive').sum(); dead=(d['status']=='Dead').sum()
    zeros_dbh=int((d['pre_DBH']==0).sum(skipna=True)) if 'pre_DBH' in d else 0
    zeros_ht =int((d['pre_HT' ]==0).sum(skipna=True)) if 'pre_HT'  in d else 0
    print(f"Saved {output_path} | rows={len(d)} | Alive={alive} | Dead={dead} | "
          f"zeros(DBH)={zeros_dbh} zeros(HT)={zeros_ht}")
    if dead==0 and (zeros_dbh>0 or zeros_ht>0) and not treat_zero_as_dead:
        print("Note: No Dead trees because zeros are considered Alive by default. "
              "Set treat_zero_as_dead=True if zeros should be marked Dead.")
    return d

clean_thinning_v3_fixed("Raw-datasets/thinning-data-3.xlsx","clean-thinning-data-3.csv")


Saved clean-thinning-data-3.csv | rows=3420 | Alive=3420 | Dead=0 | zeros(DBH)=3 zeros(HT)=0
Note: No Dead trees because zeros are considered Alive by default. Set treat_zero_as_dead=True if zeros should be marked Dead.


,Row,Tree,pre_DBH,pre_HT,pre_stem_vol,pre_stem_vol_m3,status,DBH,HT
0,1,1,8.9,51.84551,9.993395,0.282981,Alive,8.9,51.84551
1,1,2,5.7,45.20263,3.714838,0.105192,Alive,5.7,45.20263
2,1,3,6.8,47.48612,5.445394,0.154196,Alive,6.8,47.48612
3,1,4,11.0,56.20490,16.405377,0.464548,Alive,11.0,56.20490
4,1,5,9.8,53.71382,12.497137,0.353879,Alive,9.8,53.71382
...,...,...,...,...,...,...,...,...,...
3415,76,56,5.8,45.41022,3.855178,0.109166,Alive,5.8,45.41022
3416,76,57,7.6,49.14684,6.975647,0.197528,Alive,7.6,49.14684
3417,76,58,7.0,47.90130,5.805740,0.164400,Alive,7.0,47.90130
3418,76,59,9.3,52.67587,11.062618,0.313258,Alive,9.3,52.67587


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

def _find_col(cols, candidates):
    cols_l = {c.strip().lower(): c for c in cols}
    for cand in candidates:
        if cand in cols_l:
            return cols_l[cand]
    raise KeyError(f"Could not find any of: {candidates} in {list(cols)}")

def clean_xy_sheet(
    in_csv: str | Path,
    out_csv: str | Path = "dillwyn_xy_clean.csv",
    *,
    dedup: str = "first",          
    shift_row: int = 0,            
    shift_tree: int = 0,           
    verbose: bool = True
) -> pd.DataFrame:
    
    """
    - Drops rows with missing keys or coordinates
    - Rounds Row/Tree to nearest int
    - De-duplicates on (Row,Tree) via 'first' or 'mean'
    - Optional index shifts (shift_row / shift_tree)
    - Writes the cleaned file to out_csv and returns the DataFrame
    """
    in_csv = Path(in_csv)
    out_csv = Path(out_csv)

    raw = pd.read_csv(in_csv)

    # Map source column names
    row_col  = _find_col(raw.columns, ['row no.', 'row_no', 'rowno', 'row'])
    tree_col = _find_col(raw.columns, ['tree no.', 'tree_no', 'treeno', 'tree'])
    x_col    = _find_col(raw.columns, ['x'])
    y_col    = _find_col(raw.columns, ['y'])


    df = raw[[row_col, tree_col, x_col, y_col]].copy()

    # Drop rows missing keys/coords
    df = df.dropna(subset=[row_col, tree_col, x_col, y_col])

    df['Row']  = pd.to_numeric(df[row_col], errors='coerce').round().astype('Int64')
    df['Tree'] = pd.to_numeric(df[tree_col], errors='coerce').round().astype('Int64')
    df['X']    = pd.to_numeric(df[x_col], errors='coerce').astype(float)
    df['Y']    = pd.to_numeric(df[y_col], errors='coerce').astype(float)

    before = len(df)
    df = df.dropna(subset=['Row','Tree','X','Y']).copy()
    df[['Row','Tree']] = df[['Row','Tree']].astype(int)
    dropped_bad = before - len(df)

    if shift_row:
        df['Row'] += int(shift_row)
    if shift_tree:
        df['Tree'] += int(shift_tree)

    dup_mask = df.duplicated(subset=['Row','Tree'], keep=False)
    n_dups = int(dup_mask.sum())
    if n_dups and verbose:
        print(f"[clean_xy_sheet] Found {n_dups} duplicate records on (Row,Tree). Strategy = '{dedup}'.")

    if n_dups:
        if dedup == 'first':
            df = df.sort_index().drop_duplicates(subset=['Row','Tree'], keep='first')
        elif dedup == 'mean':
            df = (df.groupby(['Row','Tree'], as_index=False)
                    .agg({'X':'mean','Y':'mean'}))
        else:
            raise ValueError("dedup must be 'first' or 'mean'.")

    df = df[['Row','Tree','X','Y']].sort_values(['Row','Tree']).reset_index(drop=True)

    # Summaries
    if verbose:
        print(f"[clean_xy_sheet] Input rows            : {len(raw):,}")
        print(f"[clean_xy_sheet] Dropped NA rows        : {dropped_bad:,}")
        print(f"[clean_xy_sheet] Unique (Row,Tree) out  : {df[['Row','Tree']].drop_duplicates().shape[0]:,}")
        print(f"[clean_xy_sheet] Row range              : {df['Row'].min()} … {df['Row'].max()}")
        print(f"[clean_xy_sheet] Tree range             : {df['Tree'].min()} … {df['Tree'].max()}")
        print(f"[clean_xy_sheet] X range (m)            : {df['X'].min():.3f} … {df['X'].max():.3f}")
        print(f"[clean_xy_sheet] Y range (m)            : {df['Y'].min():.3f} … {df['Y'].max():.3f}")

    # Write file
    df.to_csv(out_csv, index=False)
    if verbose:
        print(f"[clean_xy_sheet] Wrote: {out_csv.resolve()}")

    return df

clean_xy = clean_xy_sheet(
    "data/Tree_locations-Dillwyn.csv",    # raw file
    out_csv="data/dillwyn_xy_clean.csv",  # cleaned output
    dedup="first",                   
    shift_row=0,                     
    shift_tree=0,
    verbose=True
)


[clean_xy_sheet] Input rows            : 3,629
[clean_xy_sheet] Dropped NA rows        : 0
[clean_xy_sheet] Unique (Row,Tree) out  : 3,068
[clean_xy_sheet] Row range              : 1 … 65
[clean_xy_sheet] Tree range             : 1 … 60
[clean_xy_sheet] X range (m)            : 725363.750 … 725573.820
[clean_xy_sheet] Y range (m)            : 4162506.750 … 4162747.750
[clean_xy_sheet] Wrote: /Users/amithreddy/Desktop/Forest_thinning/opt_thinning/data/dillwyn_xy_clean.csv
